# 第 3 课：从研究候选到可靠结论：搜索、反馈与独立验证

同一份资料、相同预算，多生成候选是否一定更可靠？先预测，再运行，最后用失败解释方法边界。本实验全部是人工编写的冻结响应，不测量任何模型能力。


## 运行与导航
从仓库根目录启动；无需 API key、网络或前一本 Notebook 的 kernel。读取合成 JSONL，绝不使用真实用户资料。

| 复用对象 | 输入 → 输出 | 可能错误 | 本课新增责任 |
|---|---|---|---|
| 第 2 课 HybridRetriever | query → SearchHit 列表 | 重复 source_id 被拒绝 | 将查询绑定到 claim_key |
| 第 1 课 RunLimits | 上限 → Runtime 停止条件 | 非法上限校验失败 | checkpoint 保留已用预算 |
| SourceDocument | 合成原文和人工 statement → 有类型资料 | JSON/字段错误 | 核对 key/value/quote/statement ID |

前课接口路径：`projects/capstones/research_agent/retrieval.py`、`src/agent_course/core/contracts.py`。
固定输入：`fixtures/corpus.jsonl` 和 `fixtures/search_cases.json`；项目恢复使用语料内容哈希，不仅比较文件名。

路线：react-loop → claim-graph → scaling-verification → framework-lens → research-stop。


In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from pathlib import Path
from collections.abc import Sequence
from typing import Literal
from pprint import pprint

from agent_course.core import RunLimits, StopReason
from projects.capstones.research_agent.contracts import QueryTask, ResearchRequest, SourceDocument
from projects.capstones.research_agent.corpus import load_corpus
from projects.capstones.research_agent.retrieval import HybridRetriever, contains_untrusted_instruction
from projects.capstones.research_agent.search_lab import (
    Candidate, Verdict, SearchResult, Checker, generate_candidate, load_tasks, weak_quote_check,
)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course/course-spec.json").exists())
documents = load_corpus(ROOT / "projects/capstones/research_agent/fixtures/corpus.jsonl")
retriever = HybridRetriever(documents)
print("documents:", len(documents))
print("network_calls:", 0)
print("example_hits:", [h.source_id for h in retriever.search("checkpoint typed state")])


## 1. 先看失败：真引用也能支持一个假主张
先读 dev 的第一个冻结候选。它引用了一句真实原文，却给这句话配上错误的 value。预测：只检查 quote 是否存在会怎样？直接回答是提出候选；它没有独立决定正确性的权力。


In [ ]:
task = load_tasks("dev")[0]
bad = generate_candidate(task, 0)
pprint(bad.model_dump())
print("weak_accept:", weak_quote_check(bad, documents, task.claim_key).accepted)


## 2. 先定义要回答什么，再为主张安排查询
claim 是需要核对的一条问题；query 是寻找支持的动作。不要把它们混成一个字符串。一个复杂主张可依赖更小的主张；今天先用显式顺序表达依赖，声明 checkpoint 讨论依赖先理解运行停止。循环依赖需要拒绝，不能无限规划。


In [ ]:
def plan_claims(claims: list[str], dependencies: dict[str, list[str]]) -> list[QueryTask]:
    pending = list(claims)
    done: list[str] = []
    tasks: list[QueryTask] = []
    while pending:
        ready = [key for key in pending if set(dependencies.get(key, [])) <= set(done)]
        if not ready:
            raise ValueError("claim dependency cycle or missing prerequisite")
        for key in ready:
            tasks.append(QueryTask(claim_key=key, query=key.replace("_", " "),
                                   intent="check one required claim"))
            done.append(key)
            pending.remove(key)
    return tasks

plan = plan_claims(["checkpointing", "bounded_execution"],
                  {"checkpointing": ["bounded_execution"]})
print("claim_order:", [t.claim_key for t in plan])


## 3. 计划执行与观察后改路
固定计划按顺序查询；ReAct 在行动结果回来后修改下一步。这里记录一次真实的本地检索，再从找到的不同值决定是否追加比较查询。提示词里的 Thought 不是工具结果；本课不读取隐藏推理。


In [ ]:
hits = retriever.search("bounded execution", limit=5)
catalog = {d.source_id: d for d in documents}
values = {s.value for h in hits for s in catalog[h.source_id].statements
          if s.key == "bounded_execution"
          and not contains_untrusted_instruction(catalog[h.source_id])}
next_query = "bounded execution compare authoritative sources" if len(values) > 1 else None
print("observation_values:", sorted(values))
print("next_query:", next_query)


## 4. 候选、检查器、选择器分别拥有什么权力
候选 Candidate 保存 claim_key、status、value 和 evidence；它没有可信的“自评分”。检查器读取原始语料，返回 Verdict。选择器只接受通过检查的候选；预算属于外部循环。

这是 CS329A 的计算扩展与可靠验证视角在本课中的缩小实验，不是课程论文的复现。候选覆盖率问“集合里有没有正确答案”，选择准确率问“最终选出的是否正确”。


## 5. 手写独立检查：句子存在只是第一关
按顺序核对 source 是否存在、是否允许、quote 是否在原文、statement 的 key/value/quote/ID 是否相同，再检查是否遗漏另一种说法。missing 也要核对整个小语料是否确实没有该主张。

本检查器只对人工标注的封闭语料成立。它不懂一般自然语言蕴含，也不能证明互联网没有别的来源。


In [ ]:
def check_candidate(candidate, documents, claim_key):
    # TODO: 从允许的原始 statement 重建 expected values。
    # TODO: 核对每条 evidence 的 source/key/value/quote/statement_id。
    # TODO: 区分 supported、conflict、missing，拒绝被隐藏的另一种说法。
    return Verdict(False, ("exercise_not_implemented",))


In [ ]:
good = generate_candidate(task, 1)
print("bad:", check_candidate(bad, documents, task.claim_key))
print("good:", check_candidate(good, documents, task.claim_key))


## 6. 手写选择：检查通过才有资格进入结果
选择器保持第一个通过检查的候选。这个简单规则方便我们暴露误放行：如果检查器错了，选择器不会凭空知道。反思可以建议下一次检查什么，但不能代替 Verdict。


In [ ]:
def select_candidate(incumbent, candidate, verdict):
    # TODO: incumbent 为空且 verdict.accepted 时，复制并保留候选。
    return incumbent


## 7. 手写有界搜索：生成与检查都计入预算
一个 operation unit 是一次候选提议或一次检查；六单位允许三对操作。fixed_baseline 每次读第一个冻结响应；bounded_search 沿候选列表前进。两种方法都运行到相同预算，便于做受控对照，即使已经选中也不提前结束。生产系统可提前停止，但要另报实际消耗。

这里不是 token/FLOPs/真实耗时对照。候选之间也不是独立随机抽样；不能用三次确定响应估计真实模型成功概率。


In [ ]:
def run_search(
    task: FrozenTask,
    documents: Sequence[SourceDocument],
    *,
    strategy: Literal["fixed_baseline", "bounded_search"],
    budget: int = 6,
    checker: Checker = check_candidate,
) -> SearchResult:
    """Compare two schedules under identical propose/check operation budgets.

    fixed_baseline repeats the first frozen response; bounded_search advances
    through the frozen response frontier. Both continue to the same budget for
    a controlled comparison, even if selection already has an accepted result.
    """
    if budget < 0 or strategy not in {"fixed_baseline", "bounded_search"}:
        raise ValueError("invalid experiment configuration")
    result = SearchResult(task.task_id, strategy, budget)
    for attempt in range(budget // 2):
        index = 0 if strategy == "fixed_baseline" else attempt
        candidate = generate_candidate(task, index)
        result.generation_calls += 1
        result.candidates.append(candidate)
        result.events.append({"kind": "proposal", "candidate_id": candidate.candidate_id})
        verdict = checker(candidate, documents, task.claim_key)
        result.verification_calls += 1
        result.verdicts.append(verdict)
        result.events.append({
            "kind": "verification", "accepted": verdict.accepted,
            "failures": list(verdict.failures),
        })
        result.selected = select_candidate(result.selected, candidate, verdict)
        result.events.append({
            "kind": "selection", "candidate_id": (
                result.selected.candidate_id if result.selected else None
            ), "spent": result.spent,
        })
    result.stop_reason = "budget_exhausted" if budget % 2 == 0 else "budget_remainder"
    return result


In [ ]:
for strategy in ("fixed_baseline", "bounded_search"):
    result = run_search(task, documents, strategy=strategy, budget=6)
    print(strategy, "spent:", result.spent,
          "selected:", result.selected.candidate_id if result.selected else None)
    pprint(result.events)


## 8. 开发集、保留集和错误检查器
开发集用于写规则和调试。heldout 是预先冻结的三个不同故障场景，查看之后若修改规则，就需要另建新的保留集。本课的两个集合共享语料和主张，不能据此推断跨领域泛化。人工正确性标签仅由独立评分程序读取，不传给 provider/checker/selector。

运行前预测：弱检查器能看到更多正确候选，却仍然选错吗？false-positive rate 的分母是错误候选次数；重复检查也重复计数，不是独立样本。


In [ ]:
# 保留集先只做预测；请先完成开发集，再运行自己的代码并记录结果。
heldout = load_tasks("heldout")
print("heldout_task_ids:", [item.task_id for item in heldout])
print("scope:", "toy_frozen_responses_not_model_benchmark")


## 9. 两条失败路径：篡改与冲突
真 quote 配错 value 必须失败；来源冲突不能删掉低排名的一边。对开发夹具先运行，再解释拒绝理由来自哪一层。


In [ ]:
conflict_task = load_tasks("dev")[1]
conflict = generate_candidate(conflict_task, 1)
print("full_conflict:", check_candidate(conflict, documents, conflict_task.claim_key))
conflict.evidence.pop()
print("hidden_alternative:", check_candidate(conflict, documents, conflict_task.claim_key))
unknown = good.model_copy(deep=True)
unknown.evidence[0].source_id = "not-in-corpus"
print("unknown_source:", check_candidate(unknown, documents, task.claim_key))


## 10. 从小对象映射到真实源码
只读固定 commit 中的入口和状态，不需要安装第三方框架。

| 路线 | 核心对象 | 映射到本课 | 取舍 |
|---|---|---|---|
| pi | loop、工具结果 | proposal → observation | 组合简单，持久化仍需设计 |
| Gemini CLI | 工具状态和调度器 | policy → execute | 批准状态增加复杂度 |
| Codex | turn、工具路由 | 运行范围、终结权 | 安全执行依赖环境边界 |
| Hermes | 预算和长期会话 | runtime usage、resume | 恢复不是重发原问题 |
| Grok Build | builder、生命周期 | 明确允许工具 | 构造配置不是授权证明 |
| FrontierAgent | evidence、assertion | 候选事实和汇聚 | 证据汇聚要有终结规则 |
| Claude Code Best | 工具契约 | 读写与并发属性 | 这是第三方镜像，不是官方源码 |

完整 commit、来源等级和不能推出的结论见本课 research/code-archaeology.md。


## 11. 项目运行：正常、冲突、缺失和上限
教师版运行完整例程；学生版运行公开 baseline，再按教程实现动态 frontier、verifier 和恢复。两者的职责在教程中明确区分。stop_reason 回答为什么结束，review_status 回答内容是否待复核。


In [ ]:
from projects.capstones.research_agent.starter.agent import run
request = ResearchRequest(question="How should a research agent stop and resume?",
                          required_claims=["bounded_execution", "checkpointing"])
baseline = run(request, documents)
print("review_status:", baseline.report.review_status)
print("next_tasks:", [t.query for t in baseline.next_tasks])
print("baseline_boundary:", "follow-ups proposed; resume is a learner exercise")


## 12. 最小检查与迁移题
在开发夹具上检查：错误候选不被选中；真冲突保留；预算不超额；无资料允许 missing；同样输入重放一致。修改题：给 dependency 添加循环，给预算改成奇数，给 quote 换一个存在但属于另一个 statement 的句子。

不要把“断言通过”写成真实模型能力提升。你提交的结论只覆盖这份冻结资料与规则。


## 停止策略也是学生的判断

若还有未完成主张，不要声称完成。区分已选 missing 候选与根本没有通过检查的候选；前者说明资料不足，后者在预算用完时说明需要继续工作。底层计数由脚手架提供。


In [ ]:
def stop_policy(selected, pending_claims, remaining_budget):
    # TODO: 区分缺失、未完成、预算不足和已完成。
    return "exercise_not_implemented"

print("stop_policy:", stop_policy(None, [task.claim_key], 0))


In [ ]:
# 完成函数后取消注释这些检查；错误应定位到你的函数。
# assert not check_candidate(bad, documents, task.claim_key).accepted
# assert check_candidate(good, documents, task.claim_key).accepted
# assert run_search(task, documents, strategy="bounded_search", budget=5).spent == 4
print("exercise_status:", "implement_then_check")


## 13. 三项作业与退出问题
作业全文见 `course/lessons/03-reasoning-research-agent/student/assignment.md`。

| 项 | 交付 | Notebook 入口 |
|---|---|---|
| 1 | `react_loop` 函数体不超过 30 行，交两条轨迹 | `react-loop` |
| 2 | hyperresearch 0.11.1 复现、产品对照、榜单口径 | `framework-lens` |
| 3 | 第 7 章六步框架，与 hyperresearch 对照同一任务 | `claim-graph`、`research-stop` |

本 Notebook 的检查、选择与停止函数可以承担作业三的第 4–6 步，不能代替作业一的循环。真实生成器可以替换 generate_candidate，真实检索可以替换 HybridRetriever；保留类型契约、预算和独立检查。

如果候选覆盖率从 1/3 变为 3/3，而选中准确率仍为 0，你会先检查哪一层？请用 false positive 的具体轨迹回答。

学习材料：[CS329A](https://cs329a.stanford.edu/)；[ReAct](https://arxiv.org/abs/2210.03629)；[ToT](https://arxiv.org/abs/2305.10601)；[计算分配](https://arxiv.org/abs/2408.03314)；[候选覆盖与选择](https://arxiv.org/abs/2407.21787)。
